# 실습 15 · 화학·바이오 파운데이션 모델 (ChemBERTa & ESM)
### 분자·단백질을 '언어'처럼 학습한 최신 AI 활용

**왜 중요한가 (신약개발 AI 의 현재)**
GPT 가 텍스트를 학습하듯, 최신 모델은 **분자(SMILES)와 단백질(아미노산 서열)** 을
대규모로 학습한 **파운데이션 모델**입니다. 이들은 사전학습된 '화학·생물 지식'을 담고 있어,
**적은 데이터로도 강력한 예측**을 가능하게 합니다.
- **ChemBERTa**: SMILES 를 학습한 분자 언어모델 (물성·활성 예측)
- **ESM-2**(Meta): 단백질 서열을 학습한 언어모델 (기능·구조·변이 영향 예측)

**이 노트북에서 배우는 것**
1. HuggingFace 에서 사전학습 모델 **불러와 즉시 사용**하는 법
2. ChemBERTa 로 분자 임베딩 추출 → 물성 예측
3. ESM-2 로 단백질 임베딩 추출 → 변이 영향 분석


In [ ]:
!pip install transformers torch rdkit -q
import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModel
print("Transformers 준비 완료 ✅   (GPU:", torch.cuda.is_available(), ")")

## 1. ⭐ ChemBERTa — 분자 언어모델 불러오기
HuggingFace 에 공개된 사전학습 ChemBERTa 를 **다운로드 한 줄**로 불러옵니다.
이 모델은 수백만 개 SMILES 로 사전학습되어 있습니다.


In [ ]:
# ChemBERTa 사전학습 모델 로드 (SMILES 를 이해하는 BERT)
chem_name = "DeepChem/ChemBERTa-77M-MTR"
chem_tok = AutoTokenizer.from_pretrained(chem_name)
chem_model = AutoModel.from_pretrained(chem_name).eval()
print("ChemBERTa 로드 완료 ✅")

def mol_embedding(smiles):
    """SMILES → 768차원 분자 임베딩 벡터"""
    inp = chem_tok(smiles, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        out = chem_model(**inp)
    # [CLS] 토큰 벡터 = 분자 전체의 요약 표현
    return out.last_hidden_state[:,0,:].numpy()

emb = mol_embedding("CC(=O)Oc1ccccc1C(=O)O")   # 아스피린
print("아스피린 임베딩 shape:", emb.shape, "(분자를 숫자벡터로 표현)")

## 2. ChemBERTa 임베딩으로 물성 예측
사전학습 임베딩을 특성으로 써서, **적은 코드로** 용해도를 예측합니다.
(예제 02의 직접 만든 descriptor 대신, LLM 이 학습한 표현을 사용)


In [ ]:
# 용해도 데이터 (예제 02와 동일)
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv"
df = pd.read_csv(url).rename(columns={"measured log solubility in mols per litre":"logS"})
df = df.sample(400, random_state=42).reset_index(drop=True)  # 데모용 400개

# 모든 분자의 ChemBERTa 임베딩 계산 (배치 처리)
embs = []
for i in range(0, len(df), 32):
    batch = df["smiles"].iloc[i:i+32].tolist()
    inp = chem_tok(batch, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        out = chem_model(**inp)
    embs.append(out.last_hidden_state[:,0,:].numpy())
X = np.vstack(embs); y = df["logS"].values
print("임베딩 특성 행렬:", X.shape)

In [ ]:
# 임베딩 → 용해도 회귀 (간단한 모델로도 잘 됨 = 사전학습의 힘)
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
cv = cross_val_score(RandomForestRegressor(200, random_state=42), X, y, cv=5, scoring="r2")
print("ChemBERTa 임베딩 기반 용해도 예측 R²:", round(cv.mean(),3), "±", round(cv.std(),3))
print("→ 분자 구조를 직접 손대지 않고 사전학습 표현만으로 예측 가능")

## 3. ⭐ ESM-2 — 단백질 언어모델
Meta 의 **ESM-2** 는 수억 개 단백질 서열을 학습했습니다.
아미노산 서열을 넣으면 각 잔기·전체의 **의미 벡터(임베딩)** 를 줍니다.
이 임베딩은 단백질 기능·구조·안정성 예측의 강력한 입력이 됩니다.


In [ ]:
# ESM-2 소형 모델 로드 (경량 버전, Colab 에서 실행 가능)
esm_name = "facebook/esm2_t12_35M_UR50D"
esm_tok = AutoTokenizer.from_pretrained(esm_name)
esm_model = AutoModel.from_pretrained(esm_name).eval()
print("ESM-2 로드 완료 ✅")

def protein_embedding(seq):
    inp = esm_tok(seq, return_tensors="pt")
    with torch.no_grad():
        out = esm_model(**inp)
    return out.last_hidden_state[0]   # (길이, 차원) 잔기별 임베딩

seq = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKR"
emb = protein_embedding(seq)
print("단백질 임베딩 shape:", emb.shape, "(잔기 수 × 임베딩 차원)")

## 4. ESM-2 활용 — 변이(mutation) 영향 예측
단백질의 한 아미노산이 바뀌면 기능이 크게 달라질 수 있습니다 (질병·약물저항성).
ESM-2 는 **각 위치에서 어떤 아미노산이 자연스러운지** 확률을 알고 있어,
**변이가 해로운지(비자연스러운지)** 를 점수화할 수 있습니다.


In [ ]:
from transformers import AutoModelForMaskedLM
import torch.nn.functional as F

mlm = AutoModelForMaskedLM.from_pretrained(esm_name).eval()

def mutation_score(seq, pos, wt, mut):
    """pos 위치(1부터)를 마스킹하고 wt→mut 변이의 로그확률 차이 계산.
       음수가 크면 해로운 변이 가능성 높음."""
    tokens = esm_tok(seq, return_tensors="pt")
    idx = pos  # [CLS] 때문에 위치 보정
    masked = tokens["input_ids"].clone()
    masked[0, idx] = esm_tok.mask_token_id
    with torch.no_grad():
        logits = mlm(masked, attention_mask=tokens["attention_mask"]).logits
    probs = F.log_softmax(logits[0, idx], dim=-1)
    wt_id  = esm_tok.convert_tokens_to_ids(wt)
    mut_id = esm_tok.convert_tokens_to_ids(mut)
    return (probs[mut_id] - probs[wt_id]).item()

# 예: 5번 위치 A(Ala) → 여러 변이의 영향 비교
pos = 5; wt = seq[pos-1]
print(f"{pos}번 위치 야생형(wild-type): {wt}")
for mut in ["G","V","D","P","W"]:
    s = mutation_score(seq, pos, wt, mut)
    verdict = "해로울 가능성" if s < -2 else "허용 가능성"
    print(f"  {wt}{pos}{mut}: 점수 {s:+.2f}  → {verdict}")

## 5. 화학·바이오 LLM 지형도 (참고)
| 모델 | 대상 | 용도 | 제공처 |
|---|---|---|---|
| **ChemBERTa / MolFormer** | 분자(SMILES) | 물성·활성 예측 | HuggingFace |
| **ESM-2 / ESMFold** | 단백질 서열 | 기능·구조·변이 | Meta |
| **AlphaFold2/3** | 단백질(+리간드) | 구조 예측 | DeepMind |
| **DiffDock** | 단백질+리간드 | 도킹 | MIT |
| **생성모델(REINVENT 등)** | 분자 | 신규 분자 설계 | 오픈소스 |

**공통 활용 패턴**: HuggingFace 에서 `from_pretrained` 로 불러와 → 임베딩 추출 →
소량의 자체 데이터로 fine-tune 또는 간단한 모델 연결. **적은 데이터로 높은 성능**이 강점.


## 정리 & 현장 응용
- **ChemBERTa**: 분자를 언어처럼 학습 → 임베딩으로 물성·활성 예측
- **ESM-2**: 단백질 언어모델 → 임베딩·변이 영향(단백질 공학, 항체 최적화)
- HuggingFace 로 **최신 모델을 몇 줄에 활용** — 이것이 현재 신약 AI 의 표준 작업방식
- 확장: 자체 데이터로 fine-tuning, 생성모델로 신규 분자 설계
- **면접 한 문장**: "ChemBERTa/ESM 같은 파운데이션 모델의 임베딩을 활용하면
  소량 데이터로도 물성·활성·변이영향을 예측할 수 있습니다."
